# Lab 4.5 — Self-speculation with shared KV cache

This lab adds a **KV cache** to the model so that the draft and verify forwards in self-speculation can share work. We also introduce **cache cloning** and **cache cropping** — the two operations that make NLD's self-speculation actually fast in wall-clock.

**Goals.**
1. Build a transformer that maintains a KV cache across forwards.
2. Implement `clone()` and `crop()` cache operations.
3. Run a self-spec cycle using the shared cache and measure speedup.

**Prerequisites.** Labs 4.1–4.4. Lectures 2.3 §3, 3.4 §3.


In [1]:
# Cell 1 — boilerplate
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, List

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

text = (
    "The quick brown fox jumps over the lazy dog. "
    "Pack my box with five dozen liquor jugs. "
) * 4000
chars = sorted(list(set(text))) + ["<MASK>"]
vocab_size = len(chars)
mask_token_id = vocab_size - 1
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] if i != mask_token_id else "_" for i in l)
data = torch.tensor(encode(text), dtype=torch.long)
n_train = int(0.9 * len(data))
train_data = data[:n_train]
print(f"vocab={vocab_size}")


vocab=31


/home/ubuntu/.pyenv/versions/3.12.8/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 1. The KV cache

Each layer keeps a tuple `(K, V)` of shape `(B, n_head, T_total, head_dim)`. After each forward, new K/V are appended along the sequence dimension.

Operations we'll need:
- **`append(new_k, new_v)`**: extend the cache by the new tokens.
- **`clone()`**: shallow copy that shares the underlying tensors — when one branch mutates (appends) the other is unaffected if we copy on write.
- **`crop(length)`**: truncate the cache back to `length` tokens.


In [2]:
# Cell 2 — cache class
class LayerCache:
    def __init__(self):
        self.k = None  # (B, H, T, D)
        self.v = None
    
    @property
    def length(self):
        return 0 if self.k is None else self.k.shape[2]
    
    def append(self, new_k, new_v):
        if self.k is None:
            self.k = new_k
            self.v = new_v
        else:
            self.k = torch.cat([self.k, new_k], dim=2)
            self.v = torch.cat([self.v, new_v], dim=2)
    
    def clone(self):
        new = LayerCache()
        # Share underlying storage; if subsequent appends create new tensors
        # via torch.cat, the original is unaffected (copy-on-extend).
        new.k = self.k
        new.v = self.v
        return new
    
    def crop(self, length):
        if self.k is None: return
        self.k = self.k[:, :, :length, :].contiguous()
        self.v = self.v[:, :, :length, :].contiguous()

class ModelCache:
    def __init__(self, n_layer):
        self.layers = [LayerCache() for _ in range(n_layer)]
    
    @property
    def length(self):
        return self.layers[0].length
    
    def clone(self):
        new = ModelCache.__new__(ModelCache)
        new.layers = [lc.clone() for lc in self.layers]
        return new
    
    def crop(self, length):
        for lc in self.layers:
            lc.crop(length)

# Quick sanity test
cache = ModelCache(n_layer=2)
assert cache.length == 0
print("ModelCache initialized OK.")


ModelCache initialized OK.


## 2. Causal attention with KV cache


In [3]:
# Cell 3 — model
@dataclass
class GPTConfig:
    vocab_size: int = 35
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    block_size: int = 256

class KVCachedAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head, self.n_embd = config.n_head, config.n_embd
        self.head_dim = config.n_embd // config.n_head
        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
    
    def forward(self, x, layer_cache: Optional[LayerCache] = None,
                causal: bool = True, use_cache: bool = False):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k_new = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v_new = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        
        if layer_cache is not None and layer_cache.length > 0:
            k_full = torch.cat([layer_cache.k, k_new], dim=2)
            v_full = torch.cat([layer_cache.v, v_new], dim=2)
        else:
            k_full = k_new
            v_full = v_new
        T_kv = k_full.shape[2]
        
        att = (q @ k_full.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if causal:
            # Each query at position `T_prev + i` can attend to keys up to `T_prev + i`.
            T_prev = T_kv - T
            row = torch.arange(T, device=x.device).view(-1, 1) + T_prev
            col = torch.arange(T_kv, device=x.device).view(1, -1)
            mask = (col <= row)
            att = att.masked_fill(~mask, float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v_full).transpose(1, 2).contiguous().view(B, T, C)
        
        if use_cache and layer_cache is not None:
            layer_cache.append(k_new, v_new)
        return self.proj(out)

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)
        self.proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)
    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = KVCachedAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)
    def forward(self, x, layer_cache=None, causal=True, use_cache=False):
        x = x + self.attn(self.ln1(x), layer_cache, causal=causal, use_cache=use_cache)
        return x + self.mlp(self.ln2(x))

class KVCachedGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.block_size, config.n_embd)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
    
    def forward(self, idx, cache: Optional[ModelCache] = None,
                causal: bool = True, use_cache: bool = False):
        B, T = idx.shape
        position = torch.arange(T, device=idx.device)
        if cache is not None and cache.length > 0:
            position = position + cache.length
        x = self.tok_emb(idx) + self.pos_emb(position)
        for i, blk in enumerate(self.blocks):
            lc = cache.layers[i] if cache is not None else None
            x = blk(x, layer_cache=lc, causal=causal, use_cache=use_cache)
        return self.head(self.ln_f(x))
    
    def new_cache(self):
        return ModelCache(self.config.n_layer)

config = GPTConfig(vocab_size=vocab_size, n_layer=4, n_head=4, n_embd=128,
                    block_size=256)
model = KVCachedGPT(config).to(device)
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


Params: 0.83M


## 3. Train (AR only, for simplicity)

A KV cache is most useful when the model is well-trained on AR. For pedagogical purposes we train AR-only here; in NLD the same cache works for the joint model.


In [4]:
# Cell 4 — AR training
def get_batch(batch_size=16, seq_len=64):
    ix = torch.randint(len(train_data) - seq_len, (batch_size,))
    return torch.stack([train_data[i:i+seq_len] for i in ix]).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
print("Training AR (200 steps)...")
for step in range(200):
    x = get_batch()
    logits = model(x, cache=None, causal=True, use_cache=False)
    loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)),
                            x[:, 1:].reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step % 40 == 0 or step == 199:
        print(f"  step {step}: loss {loss.item():.3f}")
print("Done.")


Training AR (200 steps)...
  step 0: loss 3.482


  step 40: loss 0.122


  step 80: loss 0.037


  step 120: loss 0.032


  step 160: loss 0.028


  step 199: loss 0.032
Done.


## 4. AR generation with KV cache vs no cache

We measure wall-clock for both. KV-cached AR generation should be much faster than re-running the full prompt each step.


In [5]:
# Cell 5 — AR with and without cache
@torch.no_grad()
def ar_no_cache(model, prompt, max_new_tokens=50):
    ids = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    for _ in range(max_new_tokens):
        idx = ids[:, -config.block_size:]
        logits = model(idx, cache=None, causal=True, use_cache=False)
        next_id = logits[0, -1].argmax().unsqueeze(0).unsqueeze(0)
        ids = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())

@torch.no_grad()
def ar_with_cache(model, prompt, max_new_tokens=50):
    ids = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    cache = model.new_cache()
    # Prefill
    logits = model(ids, cache=cache, causal=True, use_cache=True)
    next_id = logits[0, -1].argmax().unsqueeze(0).unsqueeze(0)
    ids = torch.cat([ids, next_id], dim=1)
    # Decode one token at a time
    for _ in range(max_new_tokens - 1):
        logits = model(next_id, cache=cache, causal=True, use_cache=True)
        next_id = logits[0, -1].argmax().unsqueeze(0).unsqueeze(0)
        ids = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())

model.eval()
# Warmup
_ = ar_with_cache(model, "The", max_new_tokens=5)

t0 = time.time(); out_nc = ar_no_cache(model, "The quick", max_new_tokens=50); t_nc = time.time() - t0
t0 = time.time(); out_c = ar_with_cache(model, "The quick", max_new_tokens=50); t_c = time.time() - t0
print(f"no-cache:   {t_nc:.3f}s  {50/t_nc:.0f} tok/s")
print(f"with-cache: {t_c:.3f}s  {50/t_c:.0f} tok/s")
print(f"speedup:    {t_nc/t_c:.2f}x")
print(f"\noutputs match: {out_nc == out_c}")


no-cache:   0.149s  335 tok/s
with-cache: 0.109s  459 tok/s
speedup:    1.37x

outputs match: True


## 5. The shared-cache speculation cycle

Now the key construct from Lecture 2.3:

1. **Prefill** the prompt → cache at length `L_p`.
2. **Draft** K candidate tokens by running the model forward on K tokens. This *appends* to a CLONED cache (`draft_cache`). Use causal mask. (In NLD this is the diffusion forward; we approximate with AR here so the lab focuses on the cache mechanics.)
3. **Verify** by running the same K tokens through the AR head one more time, also on a cloned cache (`verify_cache`).
4. **Accept** longest prefix where draft argmax matches verify argmax.
5. **Commit** by setting `cache = draft_cache.crop(L_p + n_accept)`.

For this lab, we'll use AR-mode for both draft (greedy K-step generate) and verify (single forward), demonstrating cache reuse.


In [6]:
# Cell 6 — shared-cache speculation
@torch.no_grad()
def shared_cache_speculation(model, prompt, max_new_tokens=50, K=4, n_cycles_max=50):
    ids = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    L_p = ids.shape[1]
    # Prefill
    base_cache = model.new_cache()
    logits = model(ids, cache=base_cache, causal=True, use_cache=True)
    first_next = logits[0, -1].argmax().unsqueeze(0).unsqueeze(0)
    ids = torch.cat([ids, first_next], dim=1)
    
    total_committed = 1
    cycles = 0
    while total_committed < max_new_tokens and cycles < n_cycles_max:
        cycles += 1
        L_curr = ids.shape[1]
        
        # DRAFT: greedy K-step generation on a cloned cache
        draft_cache = base_cache.clone()
        # We need to advance draft_cache from L_p to L_curr by processing the last L_curr - L_p tokens
        # but since we already prefilled L_p, and ids has L_p + total_committed tokens,
        # we need to feed (ids[:, L_p:L_curr]) through draft_cache (which is at length L_p).
        # Actually base_cache was prefilled at L_p, and we've appended `total_committed` tokens to ids
        # but NOT to base_cache. We need to bring base_cache forward.
        # Simpler approach: refill base_cache fully at the start of each cycle.
        # Even simpler: store base_cache at length L_curr by appending to it via the previous cycle.
        # We'll do that: each cycle commits to base_cache.
        
        # The clone above already has base_cache at length L_curr (because we update base_cache after each cycle).
        # Now draft K new tokens
        draft_input = ids[:, -1:]
        drafted = []
        for _ in range(K):
            logits = model(draft_input, cache=draft_cache, causal=True, use_cache=True)
            next_id = logits[0, -1].argmax().unsqueeze(0).unsqueeze(0)
            drafted.append(next_id)
            draft_input = next_id
        drafted = torch.cat(drafted, dim=1)  # (1, K)
        
        # VERIFY: re-run the K drafted tokens through a fresh clone, but verify all K in one forward
        verify_cache = base_cache.clone()
        verify_input = torch.cat([ids[:, -1:], drafted[:, :-1]], dim=1)  # (1, K) (previous + K-1 drafts)
        logits = model(verify_input, cache=verify_cache, causal=True, use_cache=True)
        # logits[0, i] = AR prediction at position L_curr + i (i.e., predicting drafted[i])
        verify_argmax = logits[0].argmax(dim=-1)  # (K,)
        
        # ACCEPT: longest prefix where drafted matches verify_argmax
        match = (drafted[0] == verify_argmax)
        n_accept = 0
        for i in range(K):
            if match[i]:
                n_accept += 1
            else:
                break
        # Always commit at least one (the verifier's first prediction if no draft accepted)
        if n_accept == 0:
            n_accept = 1
            accepted = verify_argmax[:1].unsqueeze(0)  # (1, 1)
            # Advance base_cache by 1 using the new token
            base_cache.crop(L_curr)  # verify_cache may be longer; ensure base is at L_curr
            _ = model(accepted, cache=base_cache, causal=True, use_cache=True)
        else:
            accepted = drafted[:, :n_accept]
            # Use verify_cache (which has K appended) — crop to L_curr + n_accept
            verify_cache.crop(L_curr + n_accept)
            base_cache = verify_cache
        
        ids = torch.cat([ids, accepted], dim=1)
        total_committed += n_accept
    
    return decode(ids[0].tolist()), cycles, total_committed

out, cycles, committed = shared_cache_speculation(model, "The quick", max_new_tokens=40, K=4)
print(out)
print(f"\ncycles={cycles}, committed={committed}, avg tokens/cycle = {committed/cycles:.2f}")


The quick brown fox jumps over the lazy dog. Pack 

cycles=10, committed=41, avg tokens/cycle = 4.10


## 6. Cache assertions

Verify the cache `clone` + `crop` operations behave correctly.


In [7]:
# Cell 7 — assertions
model.eval()
cache = model.new_cache()

# Initial state
assert cache.length == 0

# Append 10 tokens
x1 = torch.randint(0, vocab_size, (1, 10), device=device)
_ = model(x1, cache=cache, causal=True, use_cache=True)
assert cache.length == 10

# Clone
cache2 = cache.clone()
assert cache2.length == 10

# Append to clone, original should not grow
x2 = torch.randint(0, vocab_size, (1, 3), device=device)
_ = model(x2, cache=cache2, causal=True, use_cache=True)
assert cache.length == 10, f"original mutated: {cache.length}"
assert cache2.length == 13

# Crop
cache2.crop(8)
assert cache2.length == 8

print("All cache assertions passed.")


All cache assertions passed.


## 7. Recap

We have a transformer with:
- **KV cache** that maintains keys/values across forwards.
- **`clone()`** for branching off draft/verify computations.
- **`crop()`** for truncating the cache after partial-acceptance commits.
- A simple **shared-cache speculation** loop that demonstrates the mechanics.

Real NLD self-spec uses the diffusion forward (with the dual-stream mask) as the drafter, but the cache plumbing is identical. The speedup over no-cache AR is what makes self-spec viable on production hardware.

Next lab (4.6): replace the embedding for some positions with **vision tokens** to build a tiny VLM.
